In [1]:
import pandas as pd
import os
import pympi

In [16]:
sample_eaf_file_path = "/Users/manavgarg/Downloads/911/Sample/call_790SR.eaf"
sample_wav_file_path = "/Users/manavgarg/Downloads/911/Sample/call_790_CD.wav"

# Elan File Generation

In [17]:
import os
import pandas as pd
import pympi

def elan_to_dataframe(eaf_file_path):
    eaf_obj = pympi.Elan.Eaf(eaf_file_path)

    data = []

    for tier_name in eaf_obj.get_tier_names():
        annotations = eaf_obj.get_annotation_data_for_tier(tier_name)
        for (start, end, _) in annotations:
            data.append({
                "channel": tier_name,
                "startutt": start,
                "stoputt": end,
            })

    df = pd.DataFrame(data)
    return df

In [131]:
df_speaker=elan_to_dataframe("/Users/manavgarg/Downloads/911/Segmented not Transcribed Name Processed Folder/call_263_p2.eaf")

In [132]:
df_speaker

,channel,startutt,stoputt
0,operator female,2500,7810
1,operator female,9840,12900
2,operator female,14870,16490
3,operator female,28060,31890
4,operator female,32160,33440
5,operator female,36880,38600
6,operator female,52510,53670
7,operator female,60910,63580
8,operator female,65960,70950
9,operator female,77540,81940


# Sentence Only Transcription

In [19]:
import assemblyai as aai
import os
import pandas as pd
import logging
import shutil  # To handle directory deletion

aai.settings.api_key = "83df3680877f4567a3ada5b60fa61d1b"

In [23]:
def transcribe_audio_to_dataframe(folder_path, processed_audio_dir, log_dir):
    """
    Transcribes audio files in the specified folder and generates two dataframes: 
    one at the sentence level and another at the word level. Both of these dataframes will be saved as CSV
    in processed_audio_dir.

    Additionally, personal information redaction will be performed to remove sensitive data from the audio.

    The output files will have the same names as the input audio files but without their extensions.

    Args:
        folder_path (str): The path to the folder containing audio files.
        processed_audio_dir (str): The path to folder where processed audio files will be saved.
        log_dir (str): The directory where the log file will be saved.
    """
    
    # Set up logging
    log_file_path = os.path.join(log_dir, 'transcribe_audio_to_dataframe.txt')
    logging.basicConfig(filename=log_file_path, level=logging.INFO,
                        format='%(asctime)s - %(levelname)s - %(message)s')
    
    logging.info('Starting transcription process.')

    # Delete the processed_audio_dir if it exists, then create it
    if os.path.exists(processed_audio_dir):
        logging.info(f'Deleting existing directory: {processed_audio_dir}')
        shutil.rmtree(processed_audio_dir)
    
    os.makedirs(processed_audio_dir)
    logging.info(f'Created new directory: {processed_audio_dir}')

    file_names = os.listdir(folder_path)
    file_paths = [os.path.join(folder_path, file) for file in file_names]

    file_num = 1

    df_sentence_level = pd.DataFrame(columns=['CallName', 'filenum', 'channel', 'startutt', 'stoputt', 'duration', 'content'])
    df_word_level = pd.DataFrame(columns=['CallName', 'filenum', 'channel', 'startutt', 'stoputt', 'duration', 'content'])

    for idx, file_path in enumerate(file_paths):

        if not file_path.endswith(".wav"): continue

        print(file_path)

        audio_url = file_path

        config = aai.TranscriptionConfig(speaker_labels=True)

        logging.info(f'Transcribing file: {file_names[idx]}')
        transcript = aai.Transcriber().transcribe(audio_url, config)

        print(transcript.utterances)

        # Get the CallName by finding the last dot and omitting everything after it
        call_name = file_names[idx][:file_names[idx].rfind(".")]

        # print(tra)

        for utterance in transcript.utterances:
            new_row = {
                'CallName': call_name,
                'filenum': file_num,
                'channel': utterance.speaker,
                'startutt': utterance.start / 1000,
                'stoputt': utterance.end / 1000,
                'duration': utterance.end / 1000 - utterance.start / 1000,
                'content': utterance.text
            }
            df_sentence_level.loc[len(df_sentence_level)] = new_row

            for utterance_word in utterance.words:
                new_row = {
                    'CallName': call_name,
                    'filenum': file_num,
                    'channel': utterance_word.speaker,
                    'startutt': utterance_word.start / 1000,
                    'stoputt': utterance_word.end / 1000,
                    'duration': utterance_word.end / 1000 - utterance_word.start / 1000,
                    'content': utterance_word.text
                }

                df_word_level.loc[len(df_word_level)] = new_row

        file_num += 1

    # # Save the DataFrames to CSV files
    # sentence_csv_path = os.path.join(processed_audio_dir, 'sentence_level_transcription.csv')
    # word_csv_path = os.path.join(processed_audio_dir, 'word_level_transcription.csv')
    
    # df_sentence_level.to_csv(sentence_csv_path, index=False)
    # df_word_level.to_csv(word_csv_path, index=False)

    logging.info('Transcription process completed and dataframes saved.')

    return df_word_level

In [24]:
df_sentence_level = transcribe_audio_to_dataframe("/Users/manavgarg/Downloads/911/Sample/","Sample Files/processed_audio","logs")

/Users/manavgarg/Downloads/911/Sample/call_790_CD.wav
[Utterance(text="Start counting. 911, do you need police, fire or medical help? Yes, ma'am. I think my son's been killed. Where at? We're in the Evergreen Green Trailer park on Balsam street, trailer 81. Okay, what's the exact address? Walton street, trailer 81. Evergreen Trailer park in North Canton. It is in North Canton City. I'm going to give you North Canton. Stay on the line, okay? Because it's not coming up on my map. So I want to make sure you get the right place. Stand in line. They'll be right with you. North Canton Emergency. Yes, this is Judy Smith. I'm at my son's trailer park and we just broke in and it looks like he's in the bathroom, but his head's missing. But what? His head isn't there. I don't know. I know it sounds really weird, but. Wait a minute. You're in his trailer and you see him in the bathroom? It looks like him. The ladies do. I don't know. Could you just dispatch an officer? Yeah, we'll send an officer.

In [36]:
df_word_level = df_sentence_level

df_word_level['startutt']*=1000
df_word_level['stoputt']*=1000


In [26]:
df_sentence_level.head(20)

,CallName,filenum,channel,startutt,stoputt,duration,content
0,call_790_CD,1,A,6.000,6.208,0.208,Start
1,call_790_CD,1,A,6.224,6.488,0.264,counting.
2,call_790_CD,1,A,6.504,7.016,0.512,"911,"
3,call_790_CD,1,A,7.088,7.224,0.136,do
4,call_790_CD,1,A,7.232,7.304,0.072,you
5,call_790_CD,1,A,7.312,7.480,0.168,need
6,call_790_CD,1,A,7.520,7.864,0.344,"police,"
7,call_790_CD,1,A,7.952,8.200,0.248,fire
8,call_790_CD,1,A,8.240,8.440,0.200,or
9,call_790_CD,1,A,8.480,8.776,0.296,medical


# Merge Speaker

In [ ]:
import pandas as pd

def assign_words_to_speakers(df_speaker, df_word):
    df_speaker = df_speaker.sort_values(by="startutt").reset_index(drop=True)
    df_word = df_word.sort_values(by="startutt").reset_index(drop=True)

    merged_rows = []

    i, j = 0, 0
    while i < len(df_speaker) and j < len(df_word):
        speaker = df_speaker.iloc[i]
        speaker_start = speaker['startutt']
        speaker_end = speaker['stoputt']
        speaker_name = speaker['channel']

        cur_speaker_transcription = []

        while j < len(df_word):
            word = df_word.iloc[j]
            word_start = word['startutt']
            word_end = word['stoputt']
            word_text = word['content']

            if word_end <= speaker_start:
                j += 1
                continue

            if word_start >= speaker_end:
                break 
           
            if word_start < speaker_end and word_end > speaker_start:
                cur_speaker_transcription.append(word_text)

            j += 1 
        
        merged_rows.append(
            {
                'channel': speaker_name,
                'startutt': speaker_start,
                'stoputt': speaker_end,
                'content': " ".join(cur_speaker_transcription)
            }
        )

        i += 1

    df_merged = pd.DataFrame(merged_rows)
    return df_merged


In [43]:
df_merged = assign_words_to_speakers(df_speaker, df_word_level)

In [44]:
df_merged.head(20)

,channel,startutt,stoputt,content
0,operator female,5930,9130,"Start counting. 911, do you need police, fire ..."
1,caller female,10240,10710,
2,operator female,12360,13170,"Yes, ma'am."
3,caller female,13940,16480,I think my son's been killed.
4,operator female,17540,18240,Where at?
5,caller female,19010,26380,We're in the Evergreen Green Trailer park on B...
6,operator female,26490,28870,"Okay, what's the exact address?"
7,caller female,30190,35230,"Walton street, trailer 81. Evergreen Trailer p..."
8,caller female,35560,37360,It is in North Canton City. I'm
9,operator female,36640,42070,going to give you North Canton. Stay on the li...


In [46]:
df_merged.iloc[0]['content']

'Start counting. 911, do you need police, fire or medical help?'

In [48]:
df_merged.iloc[2]['content']

"Yes, ma'am."

# Mapper

In [112]:
import os
import re

def create_wav_to_eaf_mapper(folder_path):
    files = os.listdir(folder_path)

    wav_map = {}
    eaf_map = {}

    # Pattern to extract 'call_123' in a case-insensitive way
    pattern = re.compile(r'(call_\d+)', re.IGNORECASE)

    for file in files:
        match = pattern.search(file)
        if match:
            identifier = match.group(1).lower()  # normalize to lowercase
            if file.lower().endswith('.wav'):
                wav_map[identifier] = file
            elif file.lower().endswith('.eaf'):
                eaf_map[identifier] = file

    # Create final mapping: wav filename → eaf filename
    wav_to_eaf = {}
    for identifier, wav_file in wav_map.items():
        if identifier in eaf_map:
            wav_to_eaf[wav_file] = eaf_map[identifier]

    return wav_to_eaf


In [113]:
wav_to_eaf_mapper = create_wav_to_eaf_mapper('/Users/manavgarg/Downloads/911/Segmented not Transcribed Calls 1232025')

In [114]:
len(wav_to_eaf_mapper), len(wav_to_eaf_mapper.keys())

(335, 335)

In [115]:
wav_eaf_files = os.listdir('/Users/manavgarg/Downloads/911/Segmented not Transcribed Calls 1232025')

wav_files = [cur_file for cur_file in wav_eaf_files if cur_file.endswith(".wav")]

eaf_files = [cur_file for cur_file in wav_eaf_files if cur_file.endswith(".eaf")]

In [116]:
len(wav_files), len(eaf_files)

(337, 338)

In [117]:
wav_set_from_mapper = set(wav_to_eaf_mapper.keys())
wav_set_from_folder = set(wav_files)

len(wav_set_from_mapper), len(wav_set_from_folder)

(335, 337)

In [118]:
wav_set_not_in_mapper = wav_set_from_folder-wav_set_from_mapper.intersection(wav_set_from_folder)

In [119]:
wav_set_not_in_mapper

{'call_729_MG.wav', 'call_730_MG.wav'}

In [120]:
eaf_set_from_mapper = set(wav_to_eaf_mapper.values())
eaf_set_from_folder = set(eaf_files)

len(eaf_set_from_mapper), len(eaf_set_from_folder)

(335, 338)

In [121]:
eaf_set_not_in_mapper = eaf_set_from_folder-eaf_set_from_mapper.intersection(eaf_set_from_folder)

In [122]:
eaf_set_not_in_mapper

{'call_004SR.eaf', 'call_017_SR.eaf', 'call_538SR.eaf'}

In [ ]:
len()

In [125]:
import os
import re
import shutil

def standardize_and_copy_files(source_folder, target_folder, anomaly_folder):
    files = os.listdir(source_folder)
    wav_map = {}
    eaf_map = {}

    # Regex pattern to match "call_123" case-insensitively
    pattern = re.compile(r'(call_\d+)', re.IGNORECASE)

    # Step 1: Build mappings
    for file in files:
        match = pattern.search(file)
        if match:
            identifier = match.group(1).lower()
            if file.lower().endswith('.wav'):
                wav_map[identifier] = file
            elif file.lower().endswith('.eaf'):
                eaf_map[identifier] = file

    # Step 2: Create wav_to_eaf map
    wav_to_eaf = {}
    for identifier in wav_map:
        if identifier in eaf_map:
            wav_to_eaf[wav_map[identifier]] = eaf_map[identifier]

    # Step 3: Process all files
    for file in files:
        file_path = os.path.join(source_folder, file)
        match = pattern.search(file)
        if match:
            identifier = match.group(1).lower()
            ext = file.split('.')[-1].lower()
            new_name = f"{identifier}.{ext}"
            dest_path = os.path.join(target_folder, new_name)

            # If this file is part of wav_to_eaf mapping
            if (file in wav_to_eaf) or (file in wav_to_eaf.values()):
                shutil.copy(file_path, dest_path)
            else:
                shutil.copy(file_path, os.path.join(anomaly_folder, file))
        else:
            # File doesn't contain "call_{number}" → anomaly
            shutil.copy(file_path, os.path.join(anomaly_folder, file))


In [126]:
source = "/Users/manavgarg/Downloads/911/Segmented not Transcribed Calls 1232025"
target = "/Users/manavgarg/Downloads/911/Segmented not Transcribed Name Processed Folder"
anomaly = "/Users/manavgarg/Downloads/911/Anomaly Folder"

standardize_and_copy_files(source, target, anomaly)


In [133]:
len(os.listdir(target))//2

336

# Final Code

In [29]:
import os
import pandas as pd
import pympi
import logging
import assemblyai as aai
aai.settings.api_key = "83df3680877f4567a3ada5b60fa61d1b"

In [30]:
def elan_to_dataframe(folder_path):
    
    file_names = os.listdir(folder_path)
    file_paths = [os.path.join(folder_path, file) for file in file_names if file.endswith(".eaf")]
    
    data = []

    for eaf_file_path in file_paths:

        call_name = eaf_file_path.split("/")[-1].split(".")[0]

        eaf_obj = pympi.Elan.Eaf(eaf_file_path)

        for tier_name in eaf_obj.get_tier_names():
            annotations = eaf_obj.get_annotation_data_for_tier(tier_name)
            for (start, end, _) in annotations:
                data.append({
                    "channel": tier_name,
                    "startutt": start,
                    "stoputt": end,
                    "CallName": call_name
                })

    df = pd.DataFrame(data)
    return df

In [63]:
def transcribe_audio_to_dataframe(folder_path, log_dir):
    """
    Transcribes audio files in the specified folder and generates two dataframes: 
    one at the sentence level and another at the word level. Both of these dataframes will be saved as CSV
    in processed_audio_dir.

    Additionally, personal information redaction will be performed to remove sensitive data from the audio.

    The output files will have the same names as the input audio files but without their extensions.

    Args:
        folder_path (str): The path to the folder containing audio files.
        processed_audio_dir (str): The path to folder where processed audio files will be saved.
        log_dir (str): The directory where the log file will be saved.
    """
    
    # Set up logging
    log_file_path = os.path.join(log_dir, 'transcribe_audio_to_dataframe.txt')
    logging.basicConfig(filename=log_file_path, level=logging.INFO,
                        format='%(asctime)s - %(levelname)s - %(message)s')
    
    logging.info('Starting transcription process.')

    # # Delete the processed_audio_dir if it exists, then create it
    # if os.path.exists(processed_audio_dir):
    #     logging.info(f'Deleting existing directory: {processed_audio_dir}')
    #     shutil.rmtree(processed_audio_dir)
    
    # os.makedirs(processed_audio_dir)
    # logging.info(f'Created new directory: {processed_audio_dir}')

    file_names = os.listdir(folder_path)
    file_paths = [os.path.join(folder_path, file) for file in file_names]

    file_num = 1

    df_word_level = pd.DataFrame(columns=['CallName', 'filenum', 'channel', 'startutt', 'stoputt', 'duration', 'content'])

    for idx, file_path in enumerate(file_paths):

        if not file_path.endswith(".wav"): continue

        audio_url = file_path

        config = aai.TranscriptionConfig(speaker_labels=True)

        logging.info(f'Transcribing file: {file_names[idx]}')
        transcript = aai.Transcriber().transcribe(audio_url, config)

        # Get the CallName by finding the last dot and omitting everything after it
        call_name = file_names[idx][:file_names[idx].rfind(".")]

        # print(tra)

        for utterance in transcript.utterances:

            for utterance_word in utterance.words:
                new_row = {
                    'CallName': call_name,
                    'filenum': file_num,
                    'channel': utterance_word.speaker,
                    'startutt': utterance_word.start,
                    'stoputt': utterance_word.end,
                    'duration': utterance_word.end - utterance_word.start,
                    'content': utterance_word.text
                }

                df_word_level.loc[len(df_word_level)] = new_row

        file_num += 1

    logging.info('Transcription process completed and dataframes saved.')

    return df_word_level

In [ ]:
def assign_words_to_speakers_by_call(df_speaker, df_word):
    # Ensure 'CallName' is present
    assert 'CallName' in df_speaker.columns and 'CallName' in df_word.columns, \
        "'CallName' column must exist in both DataFrames"

    merged_rows = []

    # Process each call separately
    call_names = df_speaker['CallName'].unique()

    for call in call_names:
        df_s = df_speaker[df_speaker['CallName'] == call].sort_values(by="startutt").reset_index(drop=True)
        df_w = df_word[df_word['CallName'] == call].sort_values(by="startutt").reset_index(drop=True)

        i, j = 0, 0
        while i < len(df_s) and j < len(df_w):
            speaker = df_s.iloc[i]
            speaker_start = speaker['startutt']
            speaker_end = speaker['stoputt']
            speaker_name = speaker['channel']

            cur_speaker_transcription = []

            while j < len(df_w):
                word = df_w.iloc[j]
                word_start = word['startutt']
                word_end = word['stoputt']
                word_text = word['content']

                if word_end <= speaker_start:
                    j += 1
                    continue

                if word_start >= speaker_end:
                    break 
            
                if word_start < speaker_end and word_end > speaker_start:
                    cur_speaker_transcription.append(word_text)

                j += 1 
            
            merged_rows.append({
                'CallName': call,
                'channel': speaker_name,
                'startutt': speaker_start,
                'stoputt': speaker_end,
                'content': " ".join(cur_speaker_transcription)
            })
            # print(merged_rows)
            i += 1

    df_merged = pd.DataFrame(merged_rows)
    return df_merged



In [64]:
df_speaker = elan_to_dataframe("/Users/manavgarg/Downloads/911/Sample")

In [65]:
df_word_level = transcribe_audio_to_dataframe("/Users/manavgarg/Downloads/911/Sample","logs")

In [67]:
df_word_level

,CallName,filenum,channel,startutt,stoputt,duration,content
0,call_004,1,A,1760,2500,740,Emergency.
1,call_004,1,B,4560,5176,616,"Hello,"
2,call_004,1,B,5288,5512,224,this
3,call_004,1,B,5536,5624,88,is
4,call_004,1,B,5632,5752,120,the
...,...,...,...,...,...,...,...
1304,call_005,2,A,210132,210444,312,them.
1305,call_005,2,B,210532,210956,424,"Okay,"
1306,call_005,2,B,210988,211148,160,thank
1307,call_005,2,B,211164,211388,224,you.


In [68]:
df_merged = assign_words_to_speakers_by_call(df_speaker,df_word_level)

In [69]:
df_merged

,CallName,channel,startutt,stoputt,content
0,call_005,operator female,0,6010,What's the location of your emergency? Hello? ...
1,call_005,caller male,6010,6610,
2,call_005,operator female,8090,9000,how can I help you?
3,call_005,caller male,9744,11524,TJ Maxx is being robbed.
4,call_005,operator female,13750,16180,"Okay, do you see the person?"
...,...,...,...,...,...
150,call_004,caller2 female,386860,390420,Okay? The police officer is telling us to lock...
151,call_004,operator male,391744,397770,Just leave. Just stay away from that office as...
152,call_004,caller2 female,397780,399860,telling us to vacate. So should I.
153,call_004,operator male,399862,404110,Yes. If the officers are talking to you outsid...


In [72]:
df_word_level[df_word_level['CallName']=="call_005"].tail()

,CallName,filenum,channel,startutt,stoputt,duration,content
1304,call_005,2,A,210132,210444,312,them.
1305,call_005,2,B,210532,210956,424,"Okay,"
1306,call_005,2,B,210988,211148,160,thank
1307,call_005,2,B,211164,211388,224,you.
1308,call_005,2,A,211444,211628,184,Bye.


In [76]:
df_speaker[df_speaker['CallName']=="call_005"].sort_values(by='startutt').tail()

,channel,startutt,stoputt,CallName
16,operator female,204050,206960,call_005
35,caller male,207262,208630,call_005
17,operator female,208621,210360,call_005
36,caller male,210661,211470,call_005
18,operator female,211211,211951,call_005


In [74]:
df_merged[df_merged['CallName']=="call_005"].tail()

,CallName,channel,startutt,stoputt,content
32,call_005,operator female,198506,199220,
33,call_005,operator female,204050,206960,Okay. Are you out there with an officer now?
34,call_005,caller male,207262,208630,The police are here.
35,call_005,operator female,208621,210360,"Okay, I'm going to let you go. You talk to them."
36,call_005,caller male,210661,211470,"Okay, thank you. Bye."
